In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

OUTPUTS_DIR = Path("../Outputs")

LOCKED_EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
TOX_DIMENSIONS = ["toxicity", "severe_toxicity", "obscene", "identity_attack", "insult", "threat"]

# Thresholds (locked with Yiqi)
LLM_CONFIDENCE_THRESHOLD = 0.8   # LLM confidence gate
HIGH_INTENSITY_THRESHOLD = 0.5    # For proportion-of-high-intensity metric

In [2]:
# V2 routing master (with 3 new models + buckets)
master = pd.read_csv(OUTPUTS_DIR / "routed_master_v2_905.csv")

# LLM scores
llm_emo_obj = pd.read_csv(OUTPUTS_DIR / "llm_emotion_objective_905.csv")
llm_emo_youth = pd.read_csv(OUTPUTS_DIR / "llm_emotion_youth_905.csv")
llm_tox_obj = pd.read_csv(OUTPUTS_DIR / "llm_toxicity_objective_905.csv")
llm_tox_youth = pd.read_csv(OUTPUTS_DIR / "llm_toxicity_youth_905.csv")

print(f"Master (v2): {master.shape}")
print(f"LLM emotion objective: {len(llm_emo_obj)}")
print(f"LLM emotion youth: {len(llm_emo_youth)}")
print(f"LLM toxicity objective: {len(llm_tox_obj)}")
print(f"LLM toxicity youth: {len(llm_tox_youth)}")

print(f"\nBucket counts:")
print(f"  Emotion: {dict(master['emotion_bucket'].value_counts())}")
print(f"  Toxicity: {dict(master['toxicity_bucket'].value_counts())}")

Master (v2): (905, 55)
LLM emotion objective: 741
LLM emotion youth: 741
LLM toxicity objective: 84
LLM toxicity youth: 84

Bucket counts:
  Emotion: {'yellow': np.int64(653), 'gray': np.int64(156), 'green': np.int64(96)}
  Toxicity: {'green': np.int64(665), 'gray': np.int64(156), 'yellow': np.int64(83), 'red': np.int64(1)}


In [3]:
# Prepare LLM objective for merge
llm_obj_prep = llm_emo_obj[["_post_index"] + LOCKED_EMOTIONS + ["confidence", "reasoning"]].copy()
llm_obj_prep.columns = ["_post_index"] + [f"llm_obj_{e}" for e in LOCKED_EMOTIONS] + ["llm_obj_confidence", "llm_obj_reasoning"]

# Prepare LLM youth for merge
llm_youth_prep = llm_emo_youth[["_post_index"] + LOCKED_EMOTIONS + ["confidence", "reasoning"]].copy()
llm_youth_prep.columns = ["_post_index"] + [f"llm_youth_{e}" for e in LOCKED_EMOTIONS] + ["llm_youth_confidence", "llm_youth_reasoning"]

# Merge via post_index (the index of master)
master["_post_index"] = master.index
master = master.merge(llm_obj_prep, on="_post_index", how="left")
master = master.merge(llm_youth_prep, on="_post_index", how="left")

print(f"Master after LLM emotion merge: {master.shape}")
print(f"Posts with LLM objective scores: {master['llm_obj_confidence'].notna().sum()}")
print(f"Posts with LLM youth scores: {master['llm_youth_confidence'].notna().sum()}")

Master after LLM emotion merge: (905, 72)
Posts with LLM objective scores: 735
Posts with LLM youth scores: 712


In [4]:
tox_obj_cols_llm = ["overall_toxic", "insult", "threat", "identity_attack", "profanity", "harassment"]
tox_youth_cols_llm = ["feels_uncivil", "feels_insulting", "feels_threatening", "feels_attacking", "feels_profane", "feels_bullying"]

# LLM toxicity objective
llm_tox_obj_prep = llm_tox_obj[["_post_index"] + tox_obj_cols_llm + ["confidence", "reasoning"]].copy()
llm_tox_obj_prep.columns = ["_post_index"] + [f"llm_tox_obj_{c}" for c in tox_obj_cols_llm] + ["llm_tox_obj_confidence", "llm_tox_obj_reasoning"]

# LLM toxicity youth (only take columns that exist)
existing_youth_tox_cols = [c for c in tox_youth_cols_llm if c in llm_tox_youth.columns]
llm_tox_youth_prep = llm_tox_youth[["_post_index"] + existing_youth_tox_cols + ["confidence", "reasoning"]].copy()
llm_tox_youth_prep.columns = ["_post_index"] + [f"llm_tox_youth_{c}" for c in existing_youth_tox_cols] + ["llm_tox_youth_confidence", "llm_tox_youth_reasoning"]

master = master.merge(llm_tox_obj_prep, on="_post_index", how="left")
master = master.merge(llm_tox_youth_prep, on="_post_index", how="left")

print(f"Master after LLM toxicity merge: {master.shape}")

Master after LLM toxicity merge: (905, 88)


In [5]:
# For each emotion, produce ONE final score per post:
#  - If green: use MEAN of the 3 transformer models
#  - If yellow: use LLM if confidence >= 0.8, otherwise mark for human review
#  - If gray: NaN (dropped)

def get_final_emotion_score(row, emotion):
    """Return (final_score, source) tuple."""
    bucket = row["emotion_bucket"]
    
    if bucket == "gray":
        return None, "dropped"
    
    if bucket == "green":
        # Use mean of the 3 transformers
        scores = [row[f"cardiff_{emotion}"], row[f"bertweet_{emotion}"], row[f"bhadresh_{emotion}"]]
        return np.mean(scores), "transformer"
    
    if bucket in ["yellow", "red"]:
        conf = row.get("llm_obj_confidence")
        if pd.notna(conf) and conf >= LLM_CONFIDENCE_THRESHOLD:
            return row[f"llm_obj_{emotion}"], "llm_high_confidence"
        elif pd.notna(conf) and conf >= 0.2:
            # Still use LLM score but tag as medium confidence (for human review)
            return row[f"llm_obj_{emotion}"], "llm_medium_confidence"
        else:
            return None, "dropped_low_confidence"
    
    return None, "unknown"


for emotion in LOCKED_EMOTIONS:
    results = master.apply(lambda r: get_final_emotion_score(r, emotion), axis=1)
    master[f"final_{emotion}"] = [r[0] for r in results]
    master[f"final_{emotion}_source"] = [r[1] for r in results]

# Also compute confidence interval (std across the 3 transformer models per emotion)
for emotion in LOCKED_EMOTIONS:
    cols = [f"cardiff_{emotion}", f"bertweet_{emotion}", f"bhadresh_{emotion}"]
    master[f"{emotion}_std"] = master[cols].std(axis=1)
    master[f"{emotion}_transformer_mean"] = master[cols].mean(axis=1)

print("=== Emotion score sources ===")
for e in LOCKED_EMOTIONS:
    print(f"{e}:")
    print(master[f"final_{e}_source"].value_counts().to_dict())
    print()

=== Emotion score sources ===
anger:
{'llm_medium_confidence': 539, 'dropped': 156, 'transformer': 96, 'llm_high_confidence': 78, 'dropped_low_confidence': 36}

disgust:
{'llm_medium_confidence': 539, 'dropped': 156, 'transformer': 96, 'llm_high_confidence': 78, 'dropped_low_confidence': 36}

fear:
{'llm_medium_confidence': 539, 'dropped': 156, 'transformer': 96, 'llm_high_confidence': 78, 'dropped_low_confidence': 36}

joy:
{'llm_medium_confidence': 539, 'dropped': 156, 'transformer': 96, 'llm_high_confidence': 78, 'dropped_low_confidence': 36}

sadness:
{'llm_medium_confidence': 539, 'dropped': 156, 'transformer': 96, 'llm_high_confidence': 78, 'dropped_low_confidence': 36}

surprise:
{'llm_medium_confidence': 539, 'dropped': 156, 'transformer': 96, 'llm_high_confidence': 78, 'dropped_low_confidence': 36}



In [6]:
# For toxicity: use Detoxify's 6 subtypes as the transformer baseline
DETOX_COLS = [f"detox_{d}" for d in TOX_DIMENSIONS]

def get_final_toxicity_score(row, dim, detox_col):
    """Toxicity finalization based on bucket."""
    bucket = row["toxicity_bucket"]
    
    if bucket == "gray":
        return None, "dropped"
    
    if bucket == "green":
        return row[detox_col], "transformer"
    
    if bucket in ["yellow", "red"]:
        conf = row.get("llm_tox_obj_confidence")
        # Map our dimension to LLM's dimension names
        llm_col_map = {
            "toxicity": "llm_tox_obj_overall_toxic",
            "severe_toxicity": "llm_tox_obj_overall_toxic",  # LLM doesn't distinguish
            "obscene": "llm_tox_obj_profanity",
            "identity_attack": "llm_tox_obj_identity_attack",
            "insult": "llm_tox_obj_insult",
            "threat": "llm_tox_obj_threat",
        }
        llm_col = llm_col_map.get(dim)
        if llm_col and llm_col in row.index:
            if pd.notna(conf) and conf >= LLM_CONFIDENCE_THRESHOLD:
                return row[llm_col], "llm_high_confidence"
            elif pd.notna(conf) and conf >= 0.2:
                return row[llm_col], "llm_medium_confidence"
        # Fallback to transformer
        return row[detox_col], "transformer_fallback"
    
    return None, "unknown"


for dim, detox_col in zip(TOX_DIMENSIONS, DETOX_COLS):
    results = master.apply(lambda r: get_final_toxicity_score(r, dim, detox_col), axis=1)
    master[f"final_tox_{dim}"] = [r[0] for r in results]
    master[f"final_tox_{dim}_source"] = [r[1] for r in results]

print("=== Toxicity score sources ===")
for d in TOX_DIMENSIONS:
    print(f"{d}:")
    print(master[f"final_tox_{d}_source"].value_counts().to_dict())
    print()

=== Toxicity score sources ===
toxicity:
{'transformer': 665, 'dropped': 156, 'llm_high_confidence': 73, 'llm_medium_confidence': 11}

severe_toxicity:
{'transformer': 665, 'dropped': 156, 'llm_high_confidence': 73, 'llm_medium_confidence': 11}

obscene:
{'transformer': 665, 'dropped': 156, 'llm_high_confidence': 73, 'llm_medium_confidence': 11}

identity_attack:
{'transformer': 665, 'dropped': 156, 'llm_high_confidence': 73, 'llm_medium_confidence': 11}

insult:
{'transformer': 665, 'dropped': 156, 'llm_high_confidence': 73, 'llm_medium_confidence': 11}

threat:
{'transformer': 665, 'dropped': 156, 'llm_high_confidence': 73, 'llm_medium_confidence': 11}



In [7]:
# This is the backup file for the proposal
per_post_output = OUTPUTS_DIR / "final_per_post_905.csv"

# Rename student_id → pseudo_id for clarity
master["pseudo_id"] = master["student_id"]

# Reorder columns for readability
key_cols = [
    "pseudo_id", "post_index", "text_source", "text_for_analysis",
    "emotion_bucket", "toxicity_bucket",
]
final_emo_cols = [f"final_{e}" for e in LOCKED_EMOTIONS]
final_emo_sources = [f"final_{e}_source" for e in LOCKED_EMOTIONS]
emo_std_cols = [f"{e}_std" for e in LOCKED_EMOTIONS]
final_tox_cols = [f"final_tox_{d}" for d in TOX_DIMENSIONS]
final_tox_sources = [f"final_tox_{d}_source" for d in TOX_DIMENSIONS]

# Add transformer + LLM detail columns for reference
transformer_emo_cols = [f"{m}_{e}" for m in ["cardiff", "bertweet", "bhadresh"] for e in LOCKED_EMOTIONS]
llm_emo_cols = [f"llm_obj_{e}" for e in LOCKED_EMOTIONS] + ["llm_obj_confidence", "llm_obj_reasoning"]
llm_emo_youth_cols = [f"llm_youth_{e}" for e in LOCKED_EMOTIONS] + ["llm_youth_confidence", "llm_youth_reasoning"]

detox_all_cols = [f"detox_{d}" for d in TOX_DIMENSIONS]
llm_tox_cols = ["llm_tox_obj_overall_toxic", "llm_tox_obj_insult", "llm_tox_obj_threat", "llm_tox_obj_identity_attack", "llm_tox_obj_profanity", "llm_tox_obj_harassment", "llm_tox_obj_confidence", "llm_tox_obj_reasoning"]
llm_tox_cols_present = [c for c in llm_tox_cols if c in master.columns]

column_order = (
    key_cols +
    final_emo_cols + emo_std_cols + final_emo_sources +
    final_tox_cols + final_tox_sources +
    transformer_emo_cols +
    llm_emo_cols +
    llm_emo_youth_cols +
    detox_all_cols +
    llm_tox_cols_present
)
column_order = [c for c in column_order if c in master.columns]

master_out = master[column_order]
master_out.to_csv(per_post_output, index=False)

print(f"Saved: {per_post_output}")
print(f"Rows: {len(master_out)}")
print(f"Columns: {len(master_out.columns)}")

Saved: ../Outputs/final_per_post_905.csv
Rows: 905
Columns: 84


In [8]:
# Only aggregate posts that have final scores (not gray/dropped)
valid = master[master["final_joy"].notna() | master["final_tox_toxicity"].notna()].copy()

# For each student, compute per-emotion metrics
per_student_rows = []

for pseudo_id, group in valid.groupby("pseudo_id"):
    row = {"pseudo_id": pseudo_id, "post_count": len(group)}
    
    # EMOTION METRICS
    for emo in LOCKED_EMOTIONS:
        scores = group[f"final_{emo}"].dropna()
        if len(scores) > 0:
            row[f"{emo}_avg"] = scores.mean()
            row[f"{emo}_std"] = scores.std() if len(scores) > 1 else 0.0
            row[f"{emo}_min"] = scores.min()
            row[f"{emo}_max"] = scores.max()
            
            # Influence: proportion of high-intensity exposure
            high_mask = scores >= HIGH_INTENSITY_THRESHOLD
            row[f"{emo}_prop_high"] = high_mask.mean()  # 0 to 1
            
            # Influence: conditional average (only over high-intensity posts)
            if high_mask.sum() > 0:
                row[f"{emo}_avg_conditional"] = scores[high_mask].mean()
            else:
                row[f"{emo}_avg_conditional"] = np.nan
        else:
            row[f"{emo}_avg"] = np.nan
            row[f"{emo}_std"] = np.nan
            row[f"{emo}_min"] = np.nan
            row[f"{emo}_max"] = np.nan
            row[f"{emo}_prop_high"] = np.nan
            row[f"{emo}_avg_conditional"] = np.nan
    
    # TOXICITY METRICS
    for dim in TOX_DIMENSIONS:
        scores = group[f"final_tox_{dim}"].dropna()
        if len(scores) > 0:
            row[f"tox_{dim}_avg"] = scores.mean()
            row[f"tox_{dim}_std"] = scores.std() if len(scores) > 1 else 0.0
            row[f"tox_{dim}_prop_high"] = (scores >= HIGH_INTENSITY_THRESHOLD).mean()
            high_mask = scores >= HIGH_INTENSITY_THRESHOLD
            if high_mask.sum() > 0:
                row[f"tox_{dim}_avg_conditional"] = scores[high_mask].mean()
            else:
                row[f"tox_{dim}_avg_conditional"] = np.nan
        else:
            row[f"tox_{dim}_avg"] = np.nan
            row[f"tox_{dim}_std"] = np.nan
            row[f"tox_{dim}_prop_high"] = np.nan
            row[f"tox_{dim}_avg_conditional"] = np.nan
    
    per_student_rows.append(row)

per_student = pd.DataFrame(per_student_rows)

# Save
per_student_output = OUTPUTS_DIR / "final_per_student_905.csv"
per_student.to_csv(per_student_output, index=False)

print(f"Saved: {per_student_output}")
print(f"Rows: {len(per_student)}")
print(f"Columns: {len(per_student.columns)}")
print()
print("Preview:")
print(per_student[["pseudo_id", "post_count", "joy_avg", "joy_std", "joy_prop_high", "joy_avg_conditional"]].to_string())

Saved: ../Outputs/final_per_student_905.csv
Rows: 12
Columns: 62

Preview:
   pseudo_id  post_count   joy_avg   joy_std  joy_prop_high  joy_avg_conditional
0       10_A          55  0.486816  0.297233       0.490909             0.747217
1       11_A         172  0.405163  0.361723       0.329114             0.871455
2       12_A          66  0.404295  0.298573       0.433333             0.702220
3        1_A          50  0.340408  0.289385       0.250000             0.757631
4        2_A          56  0.560148  0.306162       0.589286             0.780857
5        3_A          60  0.352640  0.304261       0.305085             0.744766
6        4_A          48  0.530434  0.293739       0.531915             0.777216
7        5_A          75  0.278472  0.233139       0.222222             0.634375
8        6_A          41  0.447893  0.295349       0.457143             0.723515
9        7_A          20  0.320000  0.333404       0.350000             0.728571
10       8_A          39  0.387947

In [9]:
print("=" * 60)
print("FINAL DELIVERABLES")
print("=" * 60)
print()
print(f"1. {per_student_output}")
print(f"   → 12 rows, one per student")
print(f"   → For proposal / dashboard")
print(f"   → Includes: average, std, min, max, prop_high, avg_conditional per emotion and toxicity")
print()
print(f"2. {per_post_output}")
print(f"   → {len(master_out)} rows, one per post")
print(f"   → Backup file with full detail")
print(f"   → Includes: transformer scores, LLM scores, final consolidated, source tags")
print()

print("=== Bucket distribution (v2 routing) ===")
print(f"Emotion: {dict(master['emotion_bucket'].value_counts())}")
print(f"Toxicity: {dict(master['toxicity_bucket'].value_counts())}")

print(f"\n=== Final score source counts ===")
print("Emotion (joy example):")
print(master["final_joy_source"].value_counts().to_dict())
print("\nToxicity (overall toxicity example):")
print(master["final_tox_toxicity_source"].value_counts().to_dict())

FINAL DELIVERABLES

1. ../Outputs/final_per_student_905.csv
   → 12 rows, one per student
   → For proposal / dashboard
   → Includes: average, std, min, max, prop_high, avg_conditional per emotion and toxicity

2. ../Outputs/final_per_post_905.csv
   → 905 rows, one per post
   → Backup file with full detail
   → Includes: transformer scores, LLM scores, final consolidated, source tags

=== Bucket distribution (v2 routing) ===
Emotion: {'yellow': np.int64(653), 'gray': np.int64(156), 'green': np.int64(96)}
Toxicity: {'green': np.int64(665), 'gray': np.int64(156), 'yellow': np.int64(83), 'red': np.int64(1)}

=== Final score source counts ===
Emotion (joy example):
{'llm_medium_confidence': 539, 'dropped': 156, 'transformer': 96, 'llm_high_confidence': 78, 'dropped_low_confidence': 36}

Toxicity (overall toxicity example):
{'transformer': 665, 'dropped': 156, 'llm_high_confidence': 73, 'llm_medium_confidence': 11}


In [10]:
# Sanity check: any student with clearly weird numbers?
per_student = pd.read_csv(OUTPUTS_DIR / "final_per_student_905.csv")

print("Post count per student:")
print(per_student[["pseudo_id", "post_count"]].sort_values("post_count", ascending=False).to_string())
print()

# Check for NaN in critical columns
critical_cols = ["joy_avg", "anger_avg", "tox_toxicity_avg"]
for col in critical_cols:
    missing = per_student[col].isna().sum()
    print(f"{col} missing: {missing}/12")

Post count per student:
   pseudo_id  post_count
1       11_A         172
7        5_A          75
11       9_A          67
2       12_A          66
5        3_A          60
4        2_A          56
0       10_A          55
3        1_A          50
6        4_A          48
8        6_A          41
10       8_A          39
9        7_A          20

joy_avg missing: 0/12
anger_avg missing: 0/12
tox_toxicity_avg missing: 0/12


In [11]:
mmads = pd.read_csv(OUTPUTS_DIR / "morality_MMADS_905.csv")
print(f"MMADS shape: {mmads.shape}")
print(f"\nAll columns:")
print(list(mmads.columns))
print(f"\nFirst 3 rows:")
print(mmads.head(3))

MMADS shape: (905, 34)

All columns:
['Unnamed: 0', 'Timestamp', 'student_id', 'Q1: Which account posted this?', "Q2: Today's date", 'Q3: Which social media platform was this post from?', 'Q4: Post link/URL (in most platforms, click share, and copy URL would do the trick; if impossible to find, please put in NA here.)', 'Full Transcription', 'Q5_caption', 'Q6: Please use your own words to briefly describe the content of the image or video, if any (if no image or video, please put in NA)', 'Q7: Your reaction to the post (check all that applies)', 'Q8. Did this feel uncivil/impolite; (Uncivil = rude, insulting, demeaning, threatening, or hostile toward a person or group.)', 'Q9. What MORAL/IMMORAL ideas did the post seem to express? (Select all that apply)', 'Q10. The number of likes/favorites or equivalent engagement of the post.', 'Q11a. The number of comments of the post.', 'Q11b. The number of shares of the post.', 'Q12 (optional): Why did this post stand out to you? (1 sentence)', '

In [12]:
MORALITY_DIMENSIONS = [
    "care_virtue", "care_vice",
    "fairness_virtue", "fairness_vice",
    "loyalty_virtue", "loyalty_vice",
    "authority_virtue", "authority_vice",
    "sanctity_virtue", "sanctity_vice",
]

# Load MMADS and per-student file
mmads = pd.read_csv(OUTPUTS_DIR / "morality_MMADS_905.csv")
per_student = pd.read_csv(OUTPUTS_DIR / "final_per_student_905.csv")

# Filter MMADS to only posts with text (same filter used in emotion/toxicity)
mmads_valid = mmads[mmads["has_text"] == True].copy()

# Rename student_id to pseudo_id for consistency
mmads_valid["pseudo_id"] = mmads_valid["student_id"]

# Aggregate morality per student
morality_rows = []
for pseudo_id, group in mmads_valid.groupby("pseudo_id"):
    row = {"pseudo_id": pseudo_id}
    for dim in MORALITY_DIMENSIONS:
        col = f"mmads_{dim}"
        if col in group.columns:
            scores = group[col].dropna()
            if len(scores) > 0:
                row[f"morality_{dim}_avg"] = scores.mean()
                row[f"morality_{dim}_std"] = scores.std() if len(scores) > 1 else 0.0
                row[f"morality_{dim}_prop_high"] = (scores >= HIGH_INTENSITY_THRESHOLD).mean()
                high_mask = scores >= HIGH_INTENSITY_THRESHOLD
                if high_mask.sum() > 0:
                    row[f"morality_{dim}_avg_conditional"] = scores[high_mask].mean()
                else:
                    row[f"morality_{dim}_avg_conditional"] = np.nan
            else:
                row[f"morality_{dim}_avg"] = np.nan
                row[f"morality_{dim}_std"] = np.nan
                row[f"morality_{dim}_prop_high"] = np.nan
                row[f"morality_{dim}_avg_conditional"] = np.nan
    morality_rows.append(row)

morality_df = pd.DataFrame(morality_rows)

# Merge into per_student
per_student = per_student.merge(morality_df, on="pseudo_id", how="left")

# Save
per_student.to_csv(OUTPUTS_DIR / "final_per_student_905.csv", index=False)

print(f"Updated file: {OUTPUTS_DIR / 'final_per_student_905.csv'}")
print(f"Shape: {per_student.shape}")
print(f"\nNew morality columns added: {len([c for c in per_student.columns if 'morality_' in c])}")
print(f"\nPreview of morality (care_virtue and authority_virtue as examples):")
print(per_student[["pseudo_id", "morality_care_virtue_avg", "morality_care_virtue_prop_high", "morality_authority_virtue_avg", "morality_authority_virtue_prop_high"]].to_string(index=False))

Updated file: ../Outputs/final_per_student_905.csv
Shape: (12, 102)

New morality columns added: 40

Preview of morality (care_virtue and authority_virtue as examples):
pseudo_id  morality_care_virtue_avg  morality_care_virtue_prop_high  morality_authority_virtue_avg  morality_authority_virtue_prop_high
     10_A                  0.119903                        0.122807                       0.042955                             0.035088
     11_A                  0.162846                        0.166667                       0.093312                             0.091398
     12_A                  0.076344                        0.072464                       0.101143                             0.086957
      1_A                  0.087598                        0.087719                       0.154976                             0.157895
      2_A                  0.160502                        0.161290                       0.080059                             0.080645
      3_A      

In [13]:
# Load per-post file
per_post = pd.read_csv(OUTPUTS_DIR / "final_per_post_905.csv")

# Get MMADS columns for merging
mmads_cols_to_add = ["mmads_" + d for d in MORALITY_DIMENSIONS] + ["mmads_dominant", "mmads_dominant_score"]
mmads_subset = mmads[["student_id"] + mmads_cols_to_add].copy()

# Add post_index by matching on index (both are 0-indexed from the same source)
mmads_subset["post_index"] = mmads_subset.index

# Merge
per_post = per_post.merge(
    mmads_subset[["post_index"] + mmads_cols_to_add],
    on="post_index",
    how="left"
)

# Save
per_post.to_csv(OUTPUTS_DIR / "final_per_post_905.csv", index=False)
print(f"Updated per-post file: {per_post.shape}")
print(f"Added morality columns: {len(mmads_cols_to_add)}")

Updated per-post file: (905, 96)
Added morality columns: 12
